# Grid UCI BNN — Aggregated Results

Loads all saved `.pt` runs (grid_zigzag, grid_sticky_zigzag, grid_boomerang, grid_sticky_boomerang, nuts, nuts_horseshoe) across splits for each UCI dataset, computes RMSE, NLL, and CRPS on the held-out test set, then produces a paper-ready table (mean ± SEM across splits).

Noise is learned by every sampler in this pipeline (see `sazz/gpu_friendly/scripts/uci_bnn_grid.py`) -- there is no fixed-noise variant to filter on, unlike the old `uci_aggregated.ipynb`.

**Changes vs. the first version of this notebook** (all post-hoc on the stored samples -- no rerun, the `.pt` files are untouched):

1. The predictive distribution is now the actual posterior predictive mixture `(1/S) sum_s N(y; f_s, sigma_s^2)` rather than a single moment-matched Gaussian with `sigma` collapsed to its posterior mean. The old moment-matched NLL is kept as `NLL_legacy` so the new table can be checked against the numbers already reported.
2. Posterior `sigma` is recorded per sampler and per split, together with `RMSE / sigma` -- the diagnostic that separates "this sampler fits the mean worse" from "this sampler never drove `log_sigma` down".
3. `gradient_evals` is carried into the table as the comparable cost axis; wall time stays but is not comparable across the NumPyro/JAX and PyTorch-PDMP code paths.

Plus: the aggregate table is explicitly labelled SEM (n = number of splits), and there is a paired per-split comparison at the end -- splits share the same hard test points, so paired differences have far tighter error bars than the marginal ones.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torch.distributions import Normal
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

In [ ]:
VARIANT = "deep_narrow"
RESULTS_DIR = Path("results/paper/") / VARIANT

# Auto-discover whichever UCI datasets have actually been run so this
# notebook picks up "energy", "naval", "concrete" as soon as their results
# land, with no edits.
from sazz.gpu_friendly.scripts.uci_bnn_grid import UCI_DATASETS

DATASETS = ["boston"] #[d for d in UCI_DATASETS if (RESULTS_DIR / d).is_dir()
           # and any((RESULTS_DIR / d).glob("split_*"))]
print("Datasets found:", DATASETS)

SAMPLER_LABELS = {
    "grid_zigzag":           "Grid ZigZag",
    "grid_sticky_zigzag":    "Grid Sticky ZigZag",
    "grid_boomerang":        "Grid Boomerang",
    "grid_sticky_boomerang": "Grid Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
    #"tf_bps":                "TF BPS (Goan)",
    "tf_boomerang":          "TF Boomerang (Goan)",
}


REFERENCE_SAMPLER = "nuts"

# CRPS for a mixture predictive is O(S^2) per test point, so the exact
# version runs on a random subsample of the draws. 256 keeps it under a
# second per run while the Monte Carlo error stays well below the
# split-to-split spread.
CRPS_MIX_SUBSAMPLE = 256

## Prediction helpers

Unlike the old tree's `TorchTarget`/`ModuleGaussianLikelihood.predict()`, `BayesianModule` exposes `bm.module`/`bm.param_dict_fn` directly -- predictions go through `torch.func.functional_call`, same pattern as `grid_toy_results.ipynb`. Every run here has a trailing `log_sigma` coordinate (noise is always learned), so `predict_from_samples` doesn't need a learned/fixed branch.

`predict_from_samples` now returns the **full** `[S, N]` prediction matrix and the **per-draw** `sigma`, instead of collapsing them to `(mean, epistemic_std, scalar_noise)`. The collapse is what forced the predictive to be a single Gaussian; keeping the draws lets the metrics below use the real mixture. Every summary the old version returned (`preds.mean(0)`, `preds.std(0)`, `sigma.mean()`) is still one line away.

In [ ]:
@torch.no_grad()
def predict_from_samples(samples: Tensor, bm, X_test: Tensor) -> tuple[Tensor, Tensor]:
    """Returns (preds, sigma) with preds [S, N] and sigma [S].

    Nothing is averaged here. sigma = exp(log_sigma) per draw -- every
    sampler in this pipeline learns the noise, so samples always carries a
    trailing log_sigma column. Both are on the STANDARDIZED scale; the
    metrics below apply y_std.
    """
    weight_samples = samples[:, :-1]
    sigma = samples[:, -1].exp()

    preds = torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_test,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]

    return preds, sigma

@torch.no_grad()
def rmse_weight_mean(y_true, samples, bm, X_test, y_std) -> float:
    """RMSE of f(x, E[theta]) -- the weight-space mean network."""
    theta_bar = samples[:, :-1].mean(0)
    pred = torch.func.functional_call(bm.module, bm.param_dict_fn(theta_bar), (X_test,)).squeeze(-1)
    return float(((pred - y_true) ** 2).mean().sqrt()) * y_std

## Metrics

RMSE is unchanged -- `preds.mean(0)` is the same point estimator as before, so that column reproduces exactly.

NLL and CRPS come in two flavours:

- `nll_mixture` / `crps_mixture` evaluate the true posterior predictive `(1/S) sum_s N(y; f_s, sigma_s^2)`. This is what should be reported.
- `nll_moment_legacy` / `crps_moment` collapse to one Gaussian. `nll_moment_legacy` reproduces the *exact* old formula (including `E[sigma]` rather than `sqrt(E[sigma^2])`, which understates the noise term by Jensen) so the previous table can be verified. `crps_moment` uses the corrected second moment.

The gap between the two matters for the sampler comparison specifically: moment-matching penalises whichever posterior is least Gaussian, which is plausibly the PDMP runs. If the mixture NLL closes part of the PDMP-vs-NUTS gap, that part of the gap was the metric, not the sampler.

In [ ]:
def rmse(y_true: Tensor, preds: Tensor, y_std: float) -> float:
    mean_pred = preds.mean(0)
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std


def nll_mixture(y_true: Tensor, preds: Tensor, sigma: Tensor, y_std: float) -> float:
    """Exact posterior predictive NLL on the original scale.

    -log (1/S) sum_s N(y; f_s, sigma_s^2), computed by logsumexp over draws.
    The + log(y_std) is the change-of-variables term for the standardized ->
    original density, same as before.
    """
    S = preds.shape[0]
    lp = Normal(preds, sigma[:, None]).log_prob(y_true)      # [S, N]
    ll = torch.logsumexp(lp, dim=0) - math.log(S)            # [N]
    return float(-ll.mean() + math.log(y_std))


def nll_moment_legacy(y_true: Tensor, preds: Tensor, sigma: Tensor, y_std: float) -> float:
    """The OLD moment-matched NLL, reproduced exactly -- for comparison only.

    Single Gaussian with total_std^2 = Var_s[f_s] + (E_s[sigma_s])^2. Note
    E[sigma] rather than sqrt(E[sigma^2]): kept deliberately so this column
    reproduces the numbers already in the paper draft.
    """
    mean_pred  = preds.mean(0)
    epist_std  = preds.std(0)
    noise_std  = float(sigma.mean())
    total_std  = (epist_std ** 2 + noise_std ** 2).sqrt()
    ll = (
        -0.5 * ((y_true - mean_pred) / total_std) ** 2
        - total_std.log()
        - 0.5 * math.log(2 * math.pi)
    ).mean()
    return float(-ll + math.log(y_std))


def crps_moment(y_true: Tensor, preds: Tensor, sigma: Tensor, y_std: float) -> float:
    """Closed-form CRPS for a moment-matched Gaussian predictive, original scale.

    Differs from the old version only in using sqrt(E[sigma^2]) for the noise
    term (the correct second moment of the mixture) instead of E[sigma].
    """
    mu    = preds.mean(0) * y_std
    sigma_tot = (preds.var(0) + (sigma ** 2).mean()).sqrt() * y_std
    yt    = y_true * y_std
    d     = Normal(0.0, 1.0)
    z     = (yt - mu) / sigma_tot
    crps  = sigma_tot * (z * (2 * d.cdf(z) - 1) + 2 * d.log_prob(z).exp() - 1 / math.sqrt(math.pi))
    return float(crps.mean())


def _crps_A(m: Tensor, s: Tensor) -> Tensor:
    """E|X - m| for X ~ N(0, s^2) -- the kernel of the Gaussian-mixture CRPS."""
    d = Normal(0.0, 1.0)
    z = m / s
    return m * (2 * d.cdf(z) - 1) + 2 * s * d.log_prob(z).exp()


def crps_mixture(y_true: Tensor, preds: Tensor, sigma: Tensor, y_std: float,
                  n_sub: int = CRPS_MIX_SUBSAMPLE, seed: int = 0) -> float:
    """Exact CRPS for the Gaussian-mixture predictive, on the original scale.

    CRPS = E_s A(y - mu_s, sigma_s) - 0.5 E_{s,s'} A(mu_s - mu_s', sqrt(sigma_s^2 + sigma_s'^2))

    O(S^2) in memory, so it runs on a fixed random subsample of n_sub draws
    (same subsample seed for every sampler, so the comparison is fair).
    """
    g = torch.Generator().manual_seed(seed)
    S = preds.shape[0]
    idx = torch.randperm(S, generator=g)[:min(n_sub, S)]

    mu = preds[idx] * y_std                                   # [M, N]
    sd = (sigma[idx] * y_std)[:, None].expand_as(mu)          # [M, N]
    yt = y_true * y_std                                       # [N]

    term1 = _crps_A(yt[None, :] - mu, sd).mean(0)             # [N]
    diff  = mu[:, None, :] - mu[None, :, :]                   # [M, M, N]
    s2    = (sd[:, None, :] ** 2 + sd[None, :, :] ** 2).sqrt()
    term2 = _crps_A(diff, s2).mean((0, 1))                    # [N]

    return float((term1 - 0.5 * term2).mean())


def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())


def compute_metrics(y_true: Tensor, preds: Tensor, sigma: Tensor, y_std: float) -> dict:
    """All metrics on the original scale, plus the sigma diagnostics.

    sigma_post is the posterior mean noise sd in ORIGINAL units, and
    RMSE/sigma is the overconfidence diagnostic: ~1 means the predictive
    width matches the realised errors, >>1 means the predictive is too
    sharp, <1 means the sampler parked sigma above what the fit needs.
    """
    r     = rmse(y_true, preds, y_std)
    s_orig = float(sigma.mean()) * y_std
    return {
        "RMSE":        r,
        "NLL":         nll_mixture(y_true, preds, sigma, y_std),
        "NLL_legacy":  nll_moment_legacy(y_true, preds, sigma, y_std),
        "CRPS":        crps_mixture(y_true, preds, sigma, y_std),
        #"CRPS_moment": crps_moment(y_true, preds, sigma, y_std),
        #"coverage":    compute_coverage(y_true, preds, sigma, y_std),
        "sigma_post":  s_orig,
        "RMSE/sigma":  r / s_orig,
    }

## Load all runs and compute metrics

In [ ]:
print("Loading raw datasets for test-set reconstruction...")
raw = load_raw_datasets(tuple(DATASETS))
print("Done.", list(raw.keys()))

In [ ]:
records = []  # list of dicts: {dataset, sampler, split_id, metrics..., cost...}

@torch.no_grad()
def predict_all(bm, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]

for dataset in DATASETS:
    ds_dir = RESULTS_DIR / dataset
    if not ds_dir.exists():
        print(f"  [{dataset}] no results directory found, skipping.")
        continue
    if dataset not in raw:
        print(f"  [{dataset}] not available in raw data, skipping.")
        continue

    X_all, y_all = raw[dataset]

    split_dirs = sorted(ds_dir.glob("split_*"))
    for split_dir in split_dirs:
        split_id = int(split_dir.name.split("_")[-1])

        data = make_split(X_all, y_all, seed=BASE_SEED + split_id, dtype=DTYPE, device=DEVICE)
        X_test = data["X_test"]
        y_test = data["y_test"]
        y_std  = data["y_std"]

        # Rebuild the target ONCE per split (all runs in a split share the
        # same architecture/config -- see uci_bnn_grid.py's run_split).
        bm = None

        for pt_path in sorted(split_dir.glob("*.pt")):
            run = torch.load(pt_path, map_location="cpu", weights_only=False)
            sampler = run["sampler"]
            wall_time = run["elapsed_sec"]

            samples = run["samples"].to(dtype=DTYPE)  # [S, D]

            # Some runs (seen with tf_bps) hit their gradient budget but
            # retain zero draws -- samples is [0, D]. Nothing downstream can
            # use that, so skip it here rather than let torch.stack raise
            # "stack expects a non-empty TensorList".
            if samples.shape[0] == 0:
                print(f"  [{dataset} / split {split_id} / {sampler}] SKIP: 0 samples "
                      f"(n_events={run.get('n_events')}, grad_evals={run.get('gradient_evals')})")
                continue

            if bm is None:
                cfg = BNNConfig(
                    layer_sizes=run["layer_sizes"],
                    activation=run["activation"],
                    prior_sigma_scale=run["prior_sigma_scale"],
                )
                bm, _, _ = build_target(data, cfg)

            weight_samples = samples[:, :-1]

            preds = predict_all(bm, weight_samples, X_test)   # [S, N]
            mean_pred = preds.mean(0)
            epist_std = preds.std(0)

            noise_samples = samples[:, -1].exp()       # [S]
            noise_std_eff = float(noise_samples.mean())

            total_std = (epist_std ** 2 + noise_std_eff ** 2).sqrt()

            try:
                preds, sigma = predict_from_samples(samples, bm, X_test)
                m = compute_metrics(y_test, preds, sigma, y_std)
                m["RMSE_wmean"] = rmse_weight_mean(y_test, samples, bm, X_test, y_std)
                m["mode_ratio"] = m["RMSE_wmean"] / m["RMSE"]
                m["coverage @ 90"] = compute_coverage(y_test, mean_pred, total_std, level=0.9)
                m["coverage @ 95"] = compute_coverage(y_test, mean_pred, total_std, level=0.95)
                records.append({
                    "dataset":    dataset,
                    "sampler":    sampler,
                    "split_id":   split_id,
                    **m,
                    "grad_evals": int(run.get("gradient_evals", 0)),
                    "n_samples":  int(samples.shape[0]),
                    "wall time":  wall_time,
                })
                print(f"  [{dataset} / split {split_id} / {sampler}]  "
                      f"RMSE={m['RMSE']:.3f}  NLL={m['NLL']:.3f} "
                      f"(legacy {m['NLL_legacy']:.3f})  CRPS={m['CRPS']:.3f}  "
                      f"sigma={m['sigma_post']:.3f}  RMSE/sigma={m['RMSE/sigma']:.2f}")
            except Exception as e:
                print(f"  [{dataset} / split {split_id} / {sampler}] ERROR: {e}")

df = pd.DataFrame(records)
print(f"\nTotal records: {len(df)}")

## Aggregate: mean ± SEM across splits

The `±` is the **standard error of the mean over splits**, not the standard deviation -- with n = 5 splits, SD = SEM × sqrt(5) ≈ 2.24 × SEM. Label it as SEM wherever this table is reproduced.

`grad_evals` is reported in millions and is the cost axis to compare on; `wall time` is retained but crosses two different runtimes.

In [ ]:
def mean_sem(x):
    return x.mean(), x.sem()

METRIC_COLS = ["RMSE", "NLL", "NLL_legacy", "CRPS",
               "sigma_post", "RMSE/sigma", "RMSE_wmean", "mode_ratio",
               "coverage @ 90", "coverage @ 95"]

rows = []
for (dataset, sampler), g in df.groupby(["dataset", "sampler"]):
    row = {"Dataset": dataset, "Sampler": SAMPLER_LABELS.get(sampler, sampler),
           "n_splits": len(g)}
    for metric in METRIC_COLS:
        mu, sem = mean_sem(g[metric])
        row[metric] = f"{mu:.2f} ± {sem:.2f}"

    mu, sem = mean_sem(g["grad_evals"] / 1e6)
    row["grad evals"] = rf"{mu:.2f}$\times 10^{{6}}$" # ± {sem:.2f}"

    mu, sem = mean_sem(g["wall time"])
    mu_min = mu / 60
    if sem >= 60:
        mu_min += sem // 60
        sem_min = sem % 60
        row["wall time"] = f"{mu_min:.1f}min ± {sem_min:.1f}s"
    else:
        row["wall time"] = f"{mu_min:.1f}min ± {sem:.1f}s"

    rows.append(row)

table = pd.DataFrame(rows).set_index(["Dataset", "Sampler"])
# table

## Per-dataset tables

One metrics table per dataset, in the order runs were discovered.

`NLL` is the mixture predictive; `NLL_legacy` is the old moment-matched number. Read them side by side -- where they diverge, the old table was measuring the Gaussianity of the posterior as much as its quality.

**NUTS-HS is not a pure sampler comparison.** It swaps the fan-in Gaussian prior for a horseshoe, so any NUTS-HS-vs-grid_* difference confounds prior and sampler. Only the NUTS row is a like-for-like baseline for the PDMP samplers.

In [ ]:
per_dataset_tables = {}

for dataset in DATASETS:
    if dataset not in table.index.get_level_values("Dataset"):
        continue
    sub = table.loc[[dataset]].droplevel("Dataset")
    per_dataset_tables[dataset] = sub
    print(f"\n=== {dataset} ===")
    display(sub)

## Posterior sigma per split

The diagnostic that separates the two candidate explanations for an NLL gap that is larger than the RMSE gap:

- **sigma comparable across samplers** -> the gap is about the mean fit or the shape of the weight posterior.
- **PDMP sigma systematically above NUTS sigma** -> the PDMP chains never drove `log_sigma` down to where the data wants it, and the NLL gap is a noise-parameter mixing problem, not a statement about the predictive mean. (`log_sigma` is excluded from freezing in the sticky samplers -- `kappa = 0`, `can_freeze = False` -- and the geometry at small sigma is stiff.)

Per split, not just the mean: one split's sigma an order of magnitude off is a stuck chain, not a property of the sampler.

In [ ]:
for dataset in DATASETS:
    d = df[df["dataset"] == dataset]
    if d.empty:
        continue
    print(f"\n=== {dataset}: posterior sigma (original units) by split ===")
    piv = d.pivot(index="split_id", columns="sampler", values="sigma_post")
    piv.columns = [SAMPLER_LABELS.get(c, c) for c in piv.columns]
    display(piv.round(4))

    print(f"=== {dataset}: RMSE / sigma by split (>1 = overconfident) ===")
    piv = d.pivot(index="split_id", columns="sampler", values="RMSE/sigma")
    piv.columns = [SAMPLER_LABELS.get(c, c) for c in piv.columns]
    display(piv.round(2))

## Paired per-split comparison

Splits differ enormously in difficulty -- on a 308-row dataset a single hard test point moves the whole split -- so the marginal SEMs above overstate the uncertainty on *differences between samplers*. Every sampler saw the same five splits, so the paired difference is the right statistic.

Reported as mean ± SEM of the per-split difference `sampler - REFERENCE_SAMPLER`. Negative = better than the reference (all metrics here are lower-is-better).

In [ ]:
for dataset in DATASETS:
    d = df[df["dataset"] == dataset]
    if d.empty or REFERENCE_SAMPLER not in set(d["sampler"]):
        continue

    print(f"\n=== {dataset}: paired differences vs {SAMPLER_LABELS.get(REFERENCE_SAMPLER)} ===")
    rows = []
    for metric in ["RMSE", "NLL", "CRPS"]:
        piv = d.pivot(index="split_id", columns="sampler", values=metric)
        piv = piv.dropna()  # only splits where every sampler ran
        ref = piv[REFERENCE_SAMPLER]
        for sampler in piv.columns:
            if sampler == REFERENCE_SAMPLER:
                continue
            diff = piv[sampler] - ref
            rows.append({
                "Metric":  metric,
                "Sampler": SAMPLER_LABELS.get(sampler, sampler),
                "paired diff": f"{diff.mean():+.5f} ± {diff.sem():.5f}",
                "n": len(diff),
                "worse on": int((diff > 0).sum()),
            })
    display(pd.DataFrame(rows).set_index(["Metric", "Sampler"]))

## LaTeX export

In [ ]:
# Headline table: RMSE, exact-mixture NLL/CRPS, the overconfidence diagnostic
# RMSE/sigma_post, and wall time -- NLL_legacy, CRPS_moment, RMSE_wmean,
# mode_ratio and grad_evals stay appendix-only / for checking against the
# previous draft, not the paper table.
PAPER_COLS = ["RMSE", "NLL", "CRPS", "coverage @ 90", "RMSE/sigma", "grad evals"]

CAPTION = r"""\caption{
Predictive performance on the UCI boston test split, mean $\pm$ SEM over
5 independent train/test splits.
Each sampler contributes $S=$~8000 posterior draws $\{(\beta_s,\log\sigma_s)\}_{s=1}^S$
per split. The posterior predictive is
the $S$-component Gaussian mixture $p(y\mid x) = \frac{1}{S}\sum_{s=1}^S
\mathcal N\!\big(y;\, f_{\beta_s}(x),\, \sigma_s^2\big)$, evaluated on the held-out test.
We report $\mathrm{RMSE} = \hat\sigma_y\sqrt{\frac1N\sum_{n=1}^N\big(\bar f(x_n) - y_n\big)^2}$ with
$\bar f(x) = \frac1S\sum_s f_{\beta_s}(x)$, i.e. the RMSE of the posterior predictive mean.
\textbf{NLL} is the exact negative log density of the mixture
predictive at the test targets,
$\mathrm{NLL} = -\frac1N\sum_n \log\big(\frac1S\sum_s \mathcal N(y_n; f_{\beta_s}(x_n),
\sigma_s^2)\big)$, computed by log-sum-exp over draws.
\textbf{CRPS} is the exact Gaussian-mixture continuous ranked probability score, evaluated on a random
256-draw subsample of the $S$ posterior draws per test point.
\textbf{RMSE/$\sigma_{\mathrm{post}}$} is an overconfidence diagnostic, the point-prediction
error relative to the posterior mean noise scale $\sigma_{\mathrm{post}} =
\frac1S\sum_s\sigma_s$. Values close to $1$ means the learned observation noise
matches the realised residual scale, $\gg 1$ means the predictive is too sharp (residuals
exceed what $\sigma_{\mathrm{post}}$ would predict), $\ll 1$ means the sampler settled on
noise wider than the fit needs.
\textbf{Grad evals} is the total number of grid-node evaluations of the rate-and-derivative
closure across the run's grid-bound constructions (each window's Poisson-thinning bound
uses $n_{\mathrm{seg}}+1$ node evaluations, one target-gradient-and-directional-derivative
pair per node via forward-mode autodiff), summed over the full skeleton run and reported per
$10^6$ evaluations. The dominant per-step cost for both PDMP families here, and the
natural implementation-agnostic proxy for compute cost.
}"""
N_SAVE = 8000
for dataset, sub in per_dataset_tables.items():
    n_splits = int(sub["n_splits"].iloc[0])
    print(f"% --- {dataset} ---")
    print(f"% mean +/- SEM over {n_splits} splits")
    print(sub[PAPER_COLS].to_latex(escape=False))
    print(CAPTION.replace("\\texttt{DATASET}", dataset)
                 .replace("\\texttt{N\\_SPLITS}", str(n_splits))
                 .replace("\\texttt{N\\_SAVE}", str(N_SAVE))
                 .replace("\\texttt{BASE\\_SEED}", str(BASE_SEED)))
    print()
